# 37 — Forecast Stability

**Context Demystifies Forecasting**

Previous notebooks established:

```text
latent constraints
    >
output corrections
```

and

```text
latent_state
=
physical_state
+
residual_state
```

This notebook focuses on a single question:

> How does forecast error accumulate across horizon when latent state is constrained, weakly constrained, or unconstrained?

The emphasis is not short-horizon accuracy.

The emphasis is long-horizon stability.


## Stability hypothesis

A stable forecast should:

- maintain bounded error
- preserve latent structure
- resist drift over long horizons

We compare:

1. constrained latent trajectory
2. weakly constrained latent trajectory
3. unconstrained latent trajectory


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"

FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

rng = np.random.default_rng(260616076)


## Climate-style latent system

The latent state is represented by seasonal coordinates.

A constrained latent state remains on the physical manifold.

A weakly constrained state slowly drifts.

An unconstrained state drifts continuously.


In [ ]:
days = np.arange(0, 2000)

period = 365

latent_x = np.cos(2*np.pi*days/period)
latent_y = np.sin(2*np.pi*days/period)

physical_temperature = (
    15
    + 10*latent_y
)

residual = (
    0.8*rng.normal(size=len(days))
    + 0.8*np.sin(2*np.pi*days/37 + 0.5)
)

temperature = physical_temperature + residual

df = pd.DataFrame({
    "day": days,
    "latent_x": latent_x,
    "latent_y": latent_y,
    "physical_temperature": physical_temperature,
    "temperature": temperature,
})

df.head()


In [ ]:
split = 1200

train = df.iloc[:split]
test = df.iloc[split:]

x_test = test["day"].to_numpy()
truth = test["temperature"].to_numpy()

X_train = np.column_stack([
    np.ones(len(train)),
    train["latent_y"],
    train["latent_x"],
])

coef, *_ = np.linalg.lstsq(
    X_train,
    train["temperature"],
    rcond=None
)

coef


## Forecast trajectories

Construct three latent evolutions:

```text
constrained
weakly constrained
unconstrained
```

and decode them into temperature forecasts.


In [ ]:
phase = 2*np.pi*x_test/period

constrained_radius = np.ones_like(phase)

weak_radius = 1 + np.linspace(0, 0.25, len(phase))

unconstrained_radius = 1 + np.linspace(0, 1.0, len(phase))

constrained_x = constrained_radius*np.cos(phase)
constrained_y = constrained_radius*np.sin(phase)

weak_x = weak_radius*np.cos(phase)
weak_y = weak_radius*np.sin(phase)

unconstrained_x = unconstrained_radius*np.cos(phase)
unconstrained_y = unconstrained_radius*np.sin(phase)


In [ ]:
intercept, sin_coef, cos_coef = coef

constrained_forecast = (
    intercept
    + sin_coef*constrained_y
    + cos_coef*constrained_x
)

weak_forecast = (
    intercept
    + sin_coef*weak_y
    + cos_coef*weak_x
)

unconstrained_forecast = (
    intercept
    + sin_coef*unconstrained_y
    + cos_coef*unconstrained_x
)

forecasts = pd.DataFrame({
    "day": x_test,
    "truth": truth,
    "constrained": constrained_forecast,
    "weak": weak_forecast,
    "unconstrained": unconstrained_forecast,
})

forecasts.head()


In [ ]:
fig, ax = plt.subplots(figsize=(11,4))

ax.plot(forecasts["day"], forecasts["truth"], label="truth")
ax.plot(forecasts["day"], forecasts["constrained"], label="constrained")
ax.plot(forecasts["day"], forecasts["weak"], label="weakly constrained")
ax.plot(forecasts["day"], forecasts["unconstrained"], label="unconstrained")

ax.set_title("Forecast stability comparison")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "37_forecast_stability_comparison.png", dpi=160)
plt.show()


## Horizon error accumulation

Compute cumulative RMSE across forecast horizon.


In [ ]:
def cumulative_rmse(y_true, y_pred):
    sq = (np.asarray(y_true)-np.asarray(y_pred))**2
    return np.sqrt(
        np.cumsum(sq) /
        np.arange(1, len(sq)+1)
    )

horizon = pd.DataFrame({
    "step": np.arange(1, len(truth)+1),
    "constrained": cumulative_rmse(truth, constrained_forecast),
    "weak": cumulative_rmse(truth, weak_forecast),
    "unconstrained": cumulative_rmse(truth, unconstrained_forecast),
})

horizon.head()


In [ ]:
fig, ax = plt.subplots(figsize=(10,4))

ax.plot(horizon["step"], horizon["constrained"], label="constrained")
ax.plot(horizon["step"], horizon["weak"], label="weakly constrained")
ax.plot(horizon["step"], horizon["unconstrained"], label="unconstrained")

ax.set_title("Error accumulation across horizon")
ax.set_xlabel("forecast horizon")
ax.set_ylabel("cumulative RMSE")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "37_horizon_error.png", dpi=160)
plt.show()


## Latent drift

Measure distance from the physical manifold.

For the seasonal latent state, the physical manifold is the unit circle.


In [ ]:
latent_df = pd.DataFrame({
    "day": x_test,
    "constrained_drift": np.abs(constrained_radius - 1),
    "weak_drift": np.abs(weak_radius - 1),
    "unconstrained_drift": np.abs(unconstrained_radius - 1),
})

latent_df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(10,4))

ax.plot(latent_df["day"], latent_df["constrained_drift"], label="constrained")
ax.plot(latent_df["day"], latent_df["weak_drift"], label="weak")
ax.plot(latent_df["day"], latent_df["unconstrained_drift"], label="unconstrained")

ax.set_title("Latent drift from physical manifold")
ax.set_xlabel("day")
ax.set_ylabel("distance from manifold")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES / "37_latent_drift.png", dpi=160)
plt.show()


## Drift versus error

If latent structure matters, increasing drift should correspond to increasing forecast error.


In [ ]:
error_df = pd.DataFrame({
    "weak_drift": latent_df["weak_drift"],
    "weak_error": np.abs(truth - weak_forecast),
    "unconstrained_drift": latent_df["unconstrained_drift"],
    "unconstrained_error": np.abs(truth - unconstrained_forecast),
})

error_df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))

ax.scatter(
    error_df["unconstrained_drift"],
    error_df["unconstrained_error"],
    s=10,
    alpha=0.5,
)

ax.set_title("Forecast error versus latent drift")
ax.set_xlabel("distance from manifold")
ax.set_ylabel("absolute forecast error")

fig.tight_layout()
fig.savefig(FIGURES / "37_error_vs_drift.png", dpi=160)
plt.show()


## Stability metrics

Summarize long-horizon performance.


In [ ]:
def rmse(a,b):
    return float(
        np.sqrt(
            np.mean(
                (np.asarray(a)-np.asarray(b))**2
            )
        )
    )

metrics = pd.DataFrame([
    ["constrained", rmse(truth, constrained_forecast)],
    ["weakly_constrained", rmse(truth, weak_forecast)],
    ["unconstrained", rmse(truth, unconstrained_forecast)],
], columns=["method","RMSE"])

metrics.to_csv(
    RESULTS / "37_stability_metrics.csv",
    index=False
)

metrics


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))

ax.bar(
    metrics["method"],
    metrics["RMSE"]
)

ax.set_title("Forecast stability RMSE")

fig.tight_layout()
fig.savefig(FIGURES / "37_stability_rmse.png", dpi=160)
plt.show()


## Interpretation

The stability result can be summarized as:

```text
latent drift
    ↓
forecast drift
    ↓
error accumulation
```

Constrained latent trajectories remain close to the physical manifold and preserve forecast quality.

Weakly constrained trajectories degrade gradually.

Unconstrained trajectories accumulate error most rapidly.


## Next

Notebook 43 will synthesize the repo:

```text
Context
    ↓
Latent Constraints
    ↓
Physical + Residual Decomposition
    ↓
Interpretability
    ↓
Forecast Stability
```

and summarize the engineering lessons from all previous notebooks.
